# ColBERTv2 on LongMemEval + BEIR (T4 Colab, minimal)

Runs ColBERTv2 retrieval ONLY -- no vstash on Colab.  vstash is
evaluated on the local Mac (FastEmbed CPU + bge-small is faster
and dependency-stable there) and the resulting JSONs land in
`experiments/results/`.  This notebook produces the ColBERTv2
side; the Mac script `experiments/h2h_combine.py` merges both
into the final comparison table.

Why split the H2H this way: pylate / sentence-transformers /
torchvision on Colab churn into ABI mismatches that silently
demote vstash to FastEmbed CPU (8h instead of 9 min).  ColBERT
needs the GPU so it stays in Colab, vstash stays where it is
fastest and most stable.

Outputs (to `MyDrive/lme_h2h/`):
  - `lme_full_500_colbertv2.json` -- ColBERTv2 over LongMemEval
    (~30-45 min on T4).
  - `beir_colbertv2.json` -- ColBERTv2 over the 5 BEIR datasets
    (~15-25 min on T4).

In [ ]:
# Cell 1: Setup -- clone the experiment branch + install pylate.
# Notes on the install order, after several Colab runs proved this matters:
#
#  - pylate's transitive dep `fast-plaid` HARD-pins torch==2.9.0.  Letting
#    a normal `pip install pylate` resolve it downgrades Colab's torch
#    (2.10.0+cu128) which mismatches torchvision and breaks the entire
#    transformers / sentence-transformers import chain
#    (`operator torchvision::nms does not exist`).
#  - We don't NEED fast-plaid: it powers the PLAID indexer for large-
#    scale ColBERT search.  Our experiments build a per-question
#    in-memory index and score via `pylate.scores.colbert_scores`
#    (a plain `Q @ D.T` matmul that has no fast-plaid dependency).
#  - Solution: --no-deps for pylate, then install only the pylate
#    deps we actually exercise, with --upgrade-strategy only-if-needed
#    so we don't accidentally swap torch underneath ourselves.
BRANCH = 'feature/longmemeval-retrain-experiments'

%cd /content
!rm -rf /content/vstash
!git clone --branch $BRANCH https://github.com/stffns/vstash.git /content/vstash
%cd /content/vstash
!pip install -q -e .

# pylate without fast-plaid + the runtime deps we exercise.
!pip install -q --no-deps pylate
!pip install -q --upgrade-strategy only-if-needed huggingface_hub 'sentence-transformers>=3'

import torch
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> T4 GPU'
print('torch:', torch.__version__)
print('cuda:', torch.cuda.get_device_name(0),
      '| mem GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

# Sanity: sentence_transformers must import (this is what blows up when
# torch / torchvision get misaligned).  Then pylate.models.ColBERT.
from sentence_transformers import SentenceTransformer  # noqa: F401
from pylate import models as _pl_models  # noqa: F401
print('sentence_transformers + pylate ColBERT import OK')

In [ ]:
# Cell 2: Drive mount up front so each retrieval cell can persist its
# JSON immediately, before any runtime hiccup wipes /content/.
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/lme_h2h'
os.makedirs(DRIVE_OUT, exist_ok=True)
os.makedirs('/content/results', exist_ok=True)
print('Drive mounted, output dir:', DRIVE_OUT)

In [ ]:
# Cell 3: Download longmemeval_s (~265 MB) into the path the
# experiment script expects.
import os
from huggingface_hub import hf_hub_download
TARGET = '/content/vstash/experiments/data/longmemeval'
os.makedirs(TARGET, exist_ok=True)
p = hf_hub_download('xiaowu0162/longmemeval', 'longmemeval_s',
                    repo_type='dataset', local_dir=TARGET)
print(f'Downloaded ({os.path.getsize(p) / 1024 / 1024:.1f} MB) -> {p}')

## ColBERTv2 over LongMemEval-s

In [ ]:
# Cell 4: Per-question fresh in-memory index, encode-search.
# ~30-45 min for the full 500 questions on T4 (encoding ~50
# sessions per question dominates).
import os, time
os.chdir('/content/vstash')
t0 = time.perf_counter()
!python -m experiments.longmemeval_colbert \
    --all \
    --device cuda \
    --encode-batch-size 32 \
    --output /content/results/lme_full_500_colbertv2.json
print(f'\n[ColBERT LME] wall: {time.perf_counter() - t0:.1f}s')
!cp /content/results/lme_full_500_colbertv2.json $DRIVE_OUT/
print('Saved to Drive.')

## ColBERTv2 over BEIR (5 datasets)

In [ ]:
# Cell 5: ColBERT on the 5 BEIR datasets used in the paper.  Same
# metric definitions as experiments/beir_benchmark.py (NDCG@10,
# Recall@10, MRR), so the JSONs combine cleanly with the local
# vstash results.
import os, time
os.chdir('/content/vstash')
t0 = time.perf_counter()
!python -m experiments.beir_colbert \
    --datasets scifact nfcorpus fiqa scidocs arguana \
    --device cuda \
    --output /content/results/beir_colbertv2.json
print(f'\n[ColBERT BEIR] wall: {time.perf_counter() - t0:.1f}s')
!cp /content/results/beir_colbertv2.json $DRIVE_OUT/
print('Saved to Drive.')

In [ ]:
# Cell 6: Quick sanity print of the two outputs.
import json
lme = json.load(open('/content/results/lme_full_500_colbertv2.json'))['summary']
print('ColBERTv2 on LongMemEval-s (n=500, macro):')
for k in (1, 3, 5, 10, 20, 50):
    print(f'  R@{k:<3d} = {lme["macro"][f"recall@{k}"]:.4f}')
print()
print('ColBERTv2 on BEIR (NDCG@10):')
for r in json.load(open('/content/results/beir_colbertv2.json')):
    m = r['colbertv2']
    print(f'  {r["dataset"]:<10} NDCG@10={m["ndcg_10"]:.4f}  Recall@10={m["recall_10"]:.4f}  MRR={m["mrr"]:.4f}')
print('\nDownload these JSONs from Drive on the local Mac, then run:')
print('  python -m experiments.h2h_combine')